# Эксперименты Ames Housing (MLflow)

Все эксперименты логируются в MLflow (эксперимент `Ames_Housing_Experiments`).

Перед нужно убедиться, что сервер MLflow запущен:
```bash
$env:MLFLOW_ALLOW_FILE_STORE = "true"    
mlflow server --backend-store-uri file:./ml_flow_server/local_data --default-artifact-root file:./ml_flow_server/artifacts --host 127.0.0.1 --port 5000
```

In [ ]:
import pandas as pd
import numpy as np
import mlflow

from scipy.stats import randint, uniform

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor

import xgboost as xgb
from lightgbm import LGBMRegressor

from config import config
from src.utils import MLogger

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print(f"Tracking URI: {config.mlflow_tracking_uri}")
print(f"Experiment:   {config.experiment_name}")

Tracking URI: http://127.0.0.1:5000
Experiment:   Ames_Housing_Experiments


In [3]:
train_df = pd.read_csv('data/train.csv', na_values=['?', '-', ' ', 'N/A', 'NA', ''])
test_df = pd.read_csv('data/test.csv', na_values=['?', '-', ' ', 'N/A', 'NA', ''])

num_features = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt',
                'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
                'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr',
                'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars',
                'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
                'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

cat_features = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour',
                'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1',
                'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
                'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond',
                'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
                'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish',
                'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
                'SaleType', 'SaleCondition']

target = 'SalePrice'

In [4]:
X = train_df[num_features + cat_features]
y = train_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (1022, 79)
X_test shape: (438, 79)


In [5]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

In [6]:
logger = MLogger(
    experiment_name=config.experiment_name,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    preprocessor=preprocessor,
    tracking_uri=config.mlflow_tracking_uri,
)

## Baseline

Линейная модель

In [7]:
ridge = Ridge(alpha=1.0)

pipeline_baseline = logger.run_single(
    model=ridge,
    run_name="Baseline_Ridge"
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
2026/08/16 11:17:50 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:54 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:54 INFO mlflow.utils.environment: Detected uv project at c:\Users\matve\Desktop\ames-housing. Attempting to export requirements via 'uv export'.
2026/08/16 11:17:54 INFO mlflow.utils.uv_utils: Exported 127 dependencies via uv
2026/08/16 11:17:54 INFO mlflow.utils.environment: Successfully exported 127 requirements from uv project. Skipping package capture based inference.
2026/08/16 11:17:55 WARNING mlflow.utils.environment: 

   Запуск 'Baseline_Ridge' завершён
   Train R²: 0.9245, Test R²: 0.8822
   Train MAE: $14,030, Test MAE: $18,971
🏃 View run Baseline_Ridge at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/a2520484a484440bbe59d7a6a32460c5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


## Семейства моделей RandomForest, LGBM и XGB

In [8]:
rf_default = RandomForestRegressor(random_state=42, n_jobs=-1)

pipeline_rf = logger.run_single(
    model=rf_default,
    run_name="RF_default"
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
2026/08/16 11:17:56 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:57 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:57 INFO mlflow.utils.environment: Detected uv project at c:\Users\matve\Desktop\ames-housing. Attempting to export requirements via 'uv export'.
2026/08/16 11:17:57 INFO mlflow.utils.uv_utils: Exported 127 dependencies via uv
2026/08/16 11:17:57 INFO mlflow.utils.environment: Successfully exported 127 requirements from uv project. Skipping package capture based inference.
2026/08/16 11:17:57 WARNING mlflow.utils.environment: 

   Запуск 'RF_default' завершён
   Train R²: 0.9760, Test R²: 0.8948
   Train MAE: $6,868, Test MAE: $16,956
🏃 View run RF_default at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/a3deeacb9cb64a3293ebdf6beb51a1b2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


In [9]:
lgb_default = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

pipeline_lgb = logger.run_single(
    model=lgb_default,
    run_name="LGB_default"
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/08/16 11:17:58 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Pyth

   Запуск 'LGB_default' завершён
   Train R²: 0.9775, Test R²: 0.8992
   Train MAE: $5,011, Test MAE: $16,137
🏃 View run LGB_default at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/c51bf64c662c4a7cba08b47d95831dee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


In [10]:
xgb_default = xgb.XGBRegressor(random_state=42, n_jobs=-1)

pipeline_xgb = logger.run_single(
    model=xgb_default,
    run_name="XGB_default"
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
2026/08/16 11:17:59 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/16 11:17:59 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:17:59 INFO mlflow.utils.environ

   Запуск 'XGB_default' завершён
   Train R²: 0.9998, Test R²: 0.8955
   Train MAE: $738, Test MAE: $17,378
🏃 View run XGB_default at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/20f3cc52d305434f8b08eed2cf86b2ac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


## Подбор гиперпараметров

In [12]:
rf_param_dist = {
    'model__n_estimators': randint(50, 300),
    'model__max_depth': randint(5, 30),
    'model__max_features': uniform(0.1, 0.8),
    'model__min_samples_split': randint(2, 10),
    'model__min_samples_leaf': randint(1, 6),
}

rf_best_est, rf_best_params = logger.run_search(
    model=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    search_type='random',
    n_iter=25,
    cv=5,
    scoring='neg_mean_absolute_error',
    run_name='RF_RandomSearch_25iter'
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
2026/08/16 11:20:44 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:20:44 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:20:44 INFO mlflow.utils.environment: Detected uv project at c:\Users\matve\Desktop\ames-housing. Attempting to export requirements via 'uv export'.
2026/08/16 11:20:44 INFO mlflow.utils.uv_utils: Exported 127 dependencies via uv
2026/08/16 11:20:44 INFO mlflow.utils.environment: Successfully exported 127 requirements from uv project. Skipping package capture based inference.
2026/08/16 11:20:45 WARNING mlflow.utils.environment: 

   RandomSearch завершён для 'RF_RandomSearch_25iter'
   Лучшие параметры: {'model__max_depth': 11, 'model__max_features': 0.23641929894983324, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 138}
   Train R²: 0.9548, Test R²: 0.8929
   Train MAE: $9,101, Test MAE: $16,558
🏃 View run RF_RandomSearch_25iter at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/715278dae6b74266a6c20ddb566c3da7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


In [13]:
lgb_param_dist = {
    'model__n_estimators': randint(50, 300),
    'model__num_leaves': randint(10, 80),
    'model__learning_rate': uniform(0.01, 0.15),
    'model__max_depth': randint(3, 12),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__reg_alpha': uniform(0.0, 1.0),
    'model__reg_lambda': uniform(0.0, 1.0),
}

lgb_best_est, lgb_best_params = logger.run_search(
    model=LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=lgb_param_dist,
    search_type='random',
    n_iter=25,
    cv=5,
    scoring='neg_mean_absolute_error',
    run_name='LGB_RandomSearch_25iter'
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/08/16 11:21:27 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:21:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Pyth

   RandomSearch завершён для 'LGB_RandomSearch_25iter'
   Лучшие параметры: {'model__colsample_bytree': 0.6923575302488596, 'model__learning_rate': 0.04615381990390176, 'model__max_depth': 9, 'model__n_estimators': 221, 'model__num_leaves': 17, 'model__reg_alpha': 0.034388521115218396, 'model__reg_lambda': 0.9093204020787821, 'model__subsample': 0.7035119926400067}
   Train R²: 0.9696, Test R²: 0.9065
   Train MAE: $7,671, Test MAE: $15,609
🏃 View run LGB_RandomSearch_25iter at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/52797e3bb3a94d18917428a89f720c82
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


In [14]:
xgb_param_dist = {
    'model__n_estimators': randint(50, 300),
    'model__max_depth': randint(3, 12),
    'model__learning_rate': uniform(0.01, 0.15),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__reg_alpha': uniform(0.0, 1.0),
    'model__reg_lambda': uniform(0.0, 1.0),
}

xgb_best_est, xgb_best_params = logger.run_search(
    model=xgb.XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=xgb_param_dist,
    search_type='random',
    n_iter=25,
    cv=5,
    scoring='neg_mean_absolute_error',
    run_name='XGB_RandomSearch_25iter'
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
2026/08/16 11:32:42 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:32:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/16 11:32:42 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:32:42 INFO mlflow.utils.environ

   RandomSearch завершён для 'XGB_RandomSearch_25iter'
   Лучшие параметры: {'model__colsample_bytree': 0.8827429375390468, 'model__learning_rate': 0.11935107520614809, 'model__max_depth': 3, 'model__n_estimators': 172, 'model__reg_alpha': 0.07404465173409036, 'model__reg_lambda': 0.3584657285442726, 'model__subsample': 0.6463476238100518}
   Train R²: 0.9817, Test R²: 0.9126
   Train MAE: $7,961, Test MAE: $15,954
🏃 View run XGB_RandomSearch_25iter at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/99c31be1cfa84470b5a27fa767aaa869
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


## Шаг 4. Ансамбли

Собираем VotingRegressor и StackingRegressor из настроенных RF/LGB/XGB.
Препроцессинг применяется один раз (общий `preprocessor` пайплайна MLogger).

In [15]:
def strip_prefix(params, prefix='model__'):
    return {k[len(prefix):]: v for k, v in params.items() if k.startswith(prefix)}


rf_tuned = RandomForestRegressor(**strip_prefix(rf_best_params), random_state=42, n_jobs=-1)
lgb_tuned = LGBMRegressor(**strip_prefix(lgb_best_params), random_state=42, n_jobs=-1, verbose=-1)
xgb_tuned = xgb.XGBRegressor(**strip_prefix(xgb_best_params), random_state=42, n_jobs=-1)

voting = VotingRegressor(estimators=[
    ('rf', rf_tuned),
    ('lgb', lgb_tuned),
    ('xgb', xgb_tuned),
])

pipeline_voting = logger.run_single(
    model=voting,
    run_name='Voting_tuned'
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/08/16 11:33:14 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:33:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Pyth

   Запуск 'Voting_tuned' завершён
   Train R²: 0.9759, Test R²: 0.9143
   Train MAE: $7,605, Test MAE: $15,047
🏃 View run Voting_tuned at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/38d68dd62758456db3a63e8e8fb467f1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


In [16]:
stacking = StackingRegressor(
    estimators=[
        ('rf', rf_tuned),
        ('lgb', lgb_tuned),
        ('xgb', xgb_tuned),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1,
)

pipeline_stacking = logger.run_single(
    model=stacking,
    run_name='Stacking_tuned'
)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [11, 15, 16, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/08/16 11:33:59 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\matve\Desktop\ames-housing
2026/08/16 11:33:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Pyth

   Запуск 'Stacking_tuned' завершён
   Train R²: 0.9605, Test R²: 0.9043
   Train MAE: $8,086, Test MAE: $16,159
🏃 View run Stacking_tuned at: http://127.0.0.1:5000/#/experiments/391930776779887897/runs/1edb09b50dfb4f5c8eeb25e516511670
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/391930776779887897


## Выбор и регистрация лучшей модели

Сравниваем все запуски по `test_mae` и регистрируем лучшую модель в Model Registry.

In [21]:
runs_df = mlflow.search_runs(
    experiment_names=[config.experiment_name],
    order_by=['metrics.test_mae ASC']
)


print(runs_df[['run_id', 'tags.mlflow.runName', 'metrics.train_mae', 'metrics.test_mae', 'metrics.test_r2']].to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nЛучший запуск: {best_run['tags.mlflow.runName']} (test_mae={best_run['metrics.test_mae']:.0f})")

mlflow.register_model(
    model_uri=f"runs:/{best_run['run_id']}/model",
    name='ames_housing_best'
)

Successfully registered model 'ames_housing_best'.
2026/08/16 11:36:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 38d68dd62758456db3a63e8e8fb467f1 has no artifacts at artifact path 'model', registering model based on models:/m-ae63c9680f944a91b03c029bd327482f instead


                          run_id     tags.mlflow.runName  metrics.train_mae  metrics.test_mae  metrics.test_r2
38d68dd62758456db3a63e8e8fb467f1            Voting_tuned        7604.708008      15047.464098         0.914348
52797e3bb3a94d18917428a89f720c82 LGB_RandomSearch_25iter        7671.231801      15609.380820         0.906470
99c31be1cfa84470b5a27fa767aaa869 XGB_RandomSearch_25iter        7960.979492      15953.890625         0.912645
c51bf64c662c4a7cba08b47d95831dee             LGB_default        5011.318142      16137.302730         0.899234
97f5f478a0c8491ea3043c68646250f9             LGB_default        5011.318142      16137.302730         0.899234
1edb09b50dfb4f5c8eeb25e516511670          Stacking_tuned        8085.605736      16159.091465         0.904296
715278dae6b74266a6c20ddb566c3da7  RF_RandomSearch_25iter        9101.075499      16557.802442         0.892886
e7d2cd67535b43f9bc82a713efbb3766  RF_RandomSearch_25iter        9101.075499      16557.802442         0.892886
a

2026/08/16 11:36:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ames_housing_best, version 1
Created version '1' of model 'ames_housing_best'.


<ModelVersion: aliases=[], creation_timestamp=1786869393143, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1786869393143, metrics=None, model_id=None, name='ames_housing_best', params=None, run_id='38d68dd62758456db3a63e8e8fb467f1', run_link='', source='models:/m-ae63c9680f944a91b03c029bd327482f', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>